In [ ]:
# create model
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim

# load dataset
train_set = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transforms.ToTensor(),
)

# use DataLoader to create batches of data
batch_size = 64
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)

# create a sequential model
model = nn.Sequential(
    nn.Linear(28 * 28, 128),
    nn.ReLU(),
    nn.Linear(128, 10)
)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01)

In [ ]:
# train model
running_loss = 0.0
print("Starting training...")
model.train() # no-op now
for i, (images, labels) in enumerate(train_loader):
    flattened_images = images.view(-1, 28 * 28)
    logits = model(flattened_images)
    loss = criterion(logits, labels)

    if i == 0:
        running_loss = loss.item()
    else:
        running_loss = running_loss * 0.9 + loss.item() * 0.1
        
    if i % 100 == 0:
        print(f"  step {i}  loss={loss.item():.3f}  ema={running_loss:.3f}")

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# print the final loss
print(f"Final loss: {running_loss:.3f}")

In [ ]:
# create test dataset
test_set = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transforms.ToTensor(),
)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False)

In [ ]:
# evaluate model on one test set batch
model.eval()
correct = 0
total = 0
with torch.no_grad():
    images, labels = next(iter(test_loader))
    flattened_images = images.view(-1, 28 * 28)
    logits = model(flattened_images)
    _, predicted = torch.max(logits, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
    print(f"Test accuracy: {100 * correct / total:.2f}%")